In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score,
    precision_score, recall_score, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight


In [2]:
import os, time
OUTPUT_DIR = r'C:\Users\chand\Downloads\Telecom Churn Prediction System\outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 65)
print("  TELECOM CHURN PREDICTION — COMPLETE ML PIPELINE")
print("=" * 65)


  TELECOM CHURN PREDICTION — COMPLETE ML PIPELINE


In [4]:
# SECTION 1 — DATA LOADING & INITIAL ANALYSIS
print("\n" + "═"*65)
print("  SECTION 1: DATA LOADING & INITIAL ANALYSIS")
print("═"*65)


═════════════════════════════════════════════════════════════════
  SECTION 1: DATA LOADING & INITIAL ANALYSIS
═════════════════════════════════════════════════════════════════


In [5]:
df = pd.read_csv(r'C:\Users\chand\Downloads\Telecom Churn Prediction System\Data\telecom_churn_data.csv')
print(f"\n✔ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")



✔ Dataset loaded: 99,999 rows × 226 columns


In [6]:
# Step 1: Rename vbc_3g columns to numeric-month convention
# These 4 cols use month NAMES instead of numbers, so they are
# invisible to any loop that does endswith('_6') etc.
# Renaming makes them first-class citizens of their month groups.
df.rename(columns={
    'jun_vbc_3g': 'vbc_3g_6',   # June  → month 6
    'jul_vbc_3g': 'vbc_3g_7',   # July  → month 7
    'aug_vbc_3g': 'vbc_3g_8',   # Aug   → month 8
    'sep_vbc_3g': 'vbc_3g_9',   # Sep   → month 9 (will be dropped later)
}, inplace=True)
print("✔ Renamed vbc_3g columns to numeric-month convention (vbc_3g_6/7/8/9)")


✔ Renamed vbc_3g columns to numeric-month convention (vbc_3g_6/7/8/9)


In [7]:
# Step 2: Column groups by month suffix 
# Now vbc_3g_6/7/8/9 correctly fall into their month buckets
month_cols = {m: [c for c in df.columns if c.endswith(f'_{m}')] for m in [6, 7, 8, 9]}
date_cols  = [c for c in df.columns if 'date' in c.lower()]
id_cols    = ['mobile_number', 'circle_id']

In [8]:
# Mutually exclusive static cols: not in any month group, not IDs
all_month_cols = set(c for cols in month_cols.values() for c in cols)
static_cols    = [c for c in df.columns if c not in all_month_cols and c not in id_cols]

In [9]:
print(f"\n📅 Columns per month (after rename):")
for m, cols in month_cols.items():
    print(f"   Month {m}: {len(cols)} features")
print(f"   Date cols : {len(date_cols)}  (already inside month cols)")
print(f"   ID cols   : {len(id_cols)}")
print(f"   Static    : {len(static_cols)}  → {static_cols}")
print(f"   Total     : {len(all_month_cols) + len(id_cols) + len(static_cols)}")



📅 Columns per month (after rename):
   Month 6: 55 features
   Month 7: 55 features
   Month 8: 55 features
   Month 9: 55 features
   Date cols : 12  (already inside month cols)
   ID cols   : 2
   Static    : 4  → ['loc_og_t2o_mou', 'std_og_t2o_mou', 'loc_ic_t2o_mou', 'aon']
   Total     : 226


In [10]:
# Basic statistics 
print(f"\n📊 Basic Statistics:")
print(f"   Numeric features  : {df.select_dtypes(include='number').shape[1]}")
print(f"   Missing values    : {df.isnull().sum().sum():,}  ({df.isnull().mean().mean()*100:.1f}% overall)")
print(f"   Duplicate rows    : {df.duplicated().sum()}")



📊 Basic Statistics:
   Numeric features  : 214
   Missing values    : 3,594,931  (15.9% overall)
   Duplicate rows    : 0


In [11]:
# Missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Section 1 — Data Overview', fontsize=15, fontweight='bold')

miss_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0].head(40)
axes[0].barh(range(len(miss_pct)), miss_pct.values, color='#e74c3c', alpha=0.8)
axes[0].set_yticks(range(len(miss_pct)))
axes[0].set_yticklabels(miss_pct.index, fontsize=6)
axes[0].set_xlabel('Missing %')
axes[0].set_title(f'Top {len(miss_pct)} Columns with Missing Values')
axes[0].axvline(50, color='orange', linestyle='--', label='50% threshold')
axes[0].legend()


In [12]:
dtype_counts = df.dtypes.value_counts()
axes[1].pie(dtype_counts.values, labels=dtype_counts.index.astype(str), autopct='%1.1f%%',
            colors=['#3498db', '#2ecc71', '#e74c3c'])
axes[1].set_title('Data Types Distribution')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/1_data_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✔ Plot saved: 1_data_overview.png")



✔ Plot saved: 1_data_overview.png


In [13]:

# SECTION 2 — EXPLORATORY DATA ANALYSIS (EDA)

print("\n" + "═"*65)
print("  SECTION 2: EXPLORATORY DATA ANALYSIS (EDA)")
print("═"*65)



═════════════════════════════════════════════════════════════════
  SECTION 2: EXPLORATORY DATA ANALYSIS (EDA)
═════════════════════════════════════════════════════════════════


In [14]:
# Define Churn Label 
# High-Value Customer Churn Definition:
# A customer who had zero calls (OG + IC) AND zero recharge in Month 9
df['churn'] = (
    (df['total_og_mou_9'] == 0) &
    (df['total_ic_mou_9'] == 0) &
    (df['total_rech_amt_9'] == 0)
).astype(int)

churn_rate = df['churn'].mean() * 100
print(f"\n🎯 Target Variable: 'churn'")
print(f"   Non-Churn (0): {(df['churn']==0).sum():,}  ({100-churn_rate:.1f}%)")
print(f"   Churn     (1): {(df['churn']==1).sum():,}  ({churn_rate:.1f}%)")
print(f"   → Imbalanced dataset — churn rate = {churn_rate:.2f}%")



🎯 Target Variable: 'churn'
   Non-Churn (0): 90,979  (91.0%)
   Churn     (1): 9,020  (9.0%)
   → Imbalanced dataset — churn rate = 9.02%


In [15]:
# Month-wise usage trend 
churn_df    = df[df['churn'] == 1]
no_churn_df = df[df['churn'] == 0]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Section 2 — EDA: Usage Patterns by Churn Status', fontsize=14, fontweight='bold')

key_metrics = ['arpu', 'total_og_mou', 'total_ic_mou', 'total_rech_amt', 'vol_2g_mb', 'vol_3g_mb']
months = [6, 7, 8, 9]
colors = {'Churned': '#e74c3c', 'Not Churned': '#2ecc71'}

for ax, metric in zip(axes.flat, key_metrics):
    for label, subset, col in [('Churned', churn_df, '#e74c3c'), ('Not Churned', no_churn_df, '#2ecc71')]:
        vals = [subset[f'{metric}_{m}'].median() for m in months]
        ax.plot(months, vals, marker='o', label=label, color=col, linewidth=2)
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Month')
    ax.set_ylabel('Median Value')
    ax.legend(fontsize=8)
    ax.set_xticks(months)
    ax.set_xticklabels(['Jun', 'Jul', 'Aug', 'Sep'])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/2a_usage_trends.png', dpi=150, bbox_inches='tight')
plt.close()

In [16]:
# ARPU distribution 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Section 2 — EDA: Revenue & Recharge Analysis', fontsize=14, fontweight='bold')

for ax, m, mname in zip(axes, [6, 7, 8], ['June', 'July', 'August']):
    ax.hist(no_churn_df[f'arpu_{m}'].clip(0, 2000), bins=50, alpha=0.6, label='Not Churned', color='#2ecc71', density=True)
    ax.hist(churn_df[f'arpu_{m}'].clip(0, 2000), bins=50, alpha=0.6, label='Churned', color='#e74c3c', density=True)
    ax.set_title(f'ARPU — {mname}')
    ax.set_xlabel('ARPU (INR)')
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/2b_arpu_distribution.png', dpi=150, bbox_inches='tight')
plt.close()


In [17]:
# Churn rate by recharge amount bucket 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section 2 — EDA: Churn by Behaviour', fontsize=14, fontweight='bold')

df['rech_bucket'] = pd.cut(df['total_rech_amt_6'], bins=[0, 100, 300, 600, 1000, np.inf],
                            labels=['<100', '100-300', '300-600', '600-1000', '1000+'])
churn_by_rech = df.groupby('rech_bucket', observed=True)['churn'].mean() * 100
axes[0].bar(churn_by_rech.index.astype(str), churn_by_rech.values,
            color=['#e74c3c' if v > churn_rate else '#3498db' for v in churn_by_rech.values])
axes[0].axhline(churn_rate, color='black', linestyle='--', label=f'Avg Churn {churn_rate:.1f}%')
axes[0].set_title('Churn Rate by Recharge Amount (June)')
axes[0].set_xlabel('Recharge Bucket (INR)')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

df['aon_years'] = df['aon'] / 30 / 12
df['aon_bucket'] = pd.cut(df['aon_years'], bins=[0, 1, 2, 3, 5, np.inf],
                           labels=['<1yr', '1-2yr', '2-3yr', '3-5yr', '5+yr'])
churn_by_aon = df.groupby('aon_bucket', observed=True)['churn'].mean() * 100
axes[1].bar(churn_by_aon.index.astype(str), churn_by_aon.values,
            color=['#e74c3c' if v > churn_rate else '#3498db' for v in churn_by_aon.values])
axes[1].axhline(churn_rate, color='black', linestyle='--', label=f'Avg Churn {churn_rate:.1f}%')
axes[1].set_title('Churn Rate by Age on Network')
axes[1].set_xlabel('Years on Network')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/2c_churn_behaviour.png', dpi=150, bbox_inches='tight')
plt.close()

In [18]:
# Correlation heatmap (key features) 
key_corr_cols = ['arpu_6', 'arpu_7', 'arpu_8',
                 'total_og_mou_6', 'total_og_mou_7', 'total_og_mou_8',
                 'total_rech_amt_6', 'total_rech_amt_7', 'total_rech_amt_8',
                 'vol_3g_mb_6', 'vol_3g_mb_7', 'vol_3g_mb_8', 'churn']
corr_matrix = df[key_corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Section 2 — EDA: Correlation Heatmap (Key Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/2d_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()

In [19]:
print("\n✔ EDA Insights:")
print(f"   • Churned customers show declining ARPU from Jun→Aug (before dropping to 0 in Sep)")
print(f"   • Low-recharge customers (<100 INR) churn {churn_by_rech.iloc[0]:.1f}% of the time")
print(f"   • High-recharge customers (>1000 INR) churn only {churn_by_rech.iloc[-1]:.1f}% of the time")
print(f"   • Age on network and churn are inversely related (loyal customers churn less)")
print("✔ Plots saved: 2a, 2b, 2c, 2d")



✔ EDA Insights:
   • Churned customers show declining ARPU from Jun→Aug (before dropping to 0 in Sep)
   • Low-recharge customers (<100 INR) churn 6.8% of the time
   • High-recharge customers (>1000 INR) churn only 10.5% of the time
   • Age on network and churn are inversely related (loyal customers churn less)
✔ Plots saved: 2a, 2b, 2c, 2d


In [20]:

# SECTION 3 — FEATURE ENGINEERING

print("\n" + "═"*65)
print("  SECTION 3: FEATURE ENGINEERING")
print("═"*65)



═════════════════════════════════════════════════════════════════
  SECTION 3: FEATURE ENGINEERING
═════════════════════════════════════════════════════════════════


In [21]:
df_fe = df.copy()

In [22]:
# Drop month-9 columns (would leak the target)
# After renaming, sep_vbc_3g is now vbc_3g_9 — it is correctly
# caught here by endswith('_9') and dropped along with the rest.
m9_cols = [c for c in df_fe.columns if c.endswith('_9')]
df_fe.drop(columns=m9_cols, inplace=True)
print(f"\n✂  Dropped {len(m9_cols)} month-9 columns (includes vbc_3g_9 — prevents data leakage)")



✂  Dropped 55 month-9 columns (includes vbc_3g_9 — prevents data leakage)


In [23]:
# Drop date & ID columns 
drop_cols = id_cols + date_cols + ['rech_bucket', 'aon_bucket', 'aon_years']
drop_cols = [c for c in drop_cols if c in df_fe.columns]
df_fe.drop(columns=drop_cols, inplace=True)
print(f"✂  Dropped {len(drop_cols)} ID/date/helper columns")

✂  Dropped 14 ID/date/helper columns


In [24]:
# Derived features (trend & ratio) 
# Revenue trend: is ARPU declining? (indicator of at-risk customer)
df_fe['arpu_trend_6_to_7']   = df_fe['arpu_7']   - df_fe['arpu_6']
df_fe['arpu_trend_7_to_8']   = df_fe['arpu_8']   - df_fe['arpu_7']
df_fe['arpu_overall_trend']  = df_fe['arpu_8']   - df_fe['arpu_6']

# Call usage trend
df_fe['og_trend_6_to_7'] = df_fe['total_og_mou_7'] - df_fe['total_og_mou_6']
df_fe['og_trend_7_to_8'] = df_fe['total_og_mou_8'] - df_fe['total_og_mou_7']
df_fe['og_overall_trend']= df_fe['total_og_mou_8'] - df_fe['total_og_mou_6']

df_fe['ic_trend_6_to_7'] = df_fe['total_ic_mou_7'] - df_fe['total_ic_mou_6']
df_fe['ic_trend_7_to_8'] = df_fe['total_ic_mou_8'] - df_fe['total_ic_mou_7']
df_fe['ic_overall_trend']= df_fe['total_ic_mou_8'] - df_fe['total_ic_mou_6']

# Recharge trend
df_fe['rech_trend_6_to_7'] = df_fe['total_rech_amt_7'] - df_fe['total_rech_amt_6']
df_fe['rech_trend_7_to_8'] = df_fe['total_rech_amt_8'] - df_fe['total_rech_amt_7']
df_fe['rech_overall_trend']= df_fe['total_rech_amt_8'] - df_fe['total_rech_amt_6']


In [25]:
# Average features across months 6-8
# vbc_3g_6/7/8 are now included because the rename made them
# match the f'{metric}_{m}' pattern used inside this loop.
for metric in ['arpu', 'total_og_mou', 'total_ic_mou', 'total_rech_amt',
               'vol_2g_mb', 'vol_3g_mb', 'vbc_3g']:
    cols = [f'{metric}_{m}' for m in [6, 7, 8] if f'{metric}_{m}' in df_fe.columns]
    if cols:
        df_fe[f'{metric}_avg_678'] = df_fe[cols].mean(axis=1)
        df_fe[f'{metric}_std_678'] = df_fe[cols].std(axis=1)

# VBC-3G trend (voice-before-consume — measures 3G pack burn-rate)
# A rising VBC means the customer is consuming data faster than they
# recharge; a sudden drop often precedes churn.
if 'vbc_3g_6' in df_fe.columns and 'vbc_3g_8' in df_fe.columns:
    df_fe['vbc_3g_trend_6_to_7']  = df_fe['vbc_3g_7'] - df_fe['vbc_3g_6']
    df_fe['vbc_3g_trend_7_to_8']  = df_fe['vbc_3g_8'] - df_fe['vbc_3g_7']
    df_fe['vbc_3g_overall_trend'] = df_fe['vbc_3g_8'] - df_fe['vbc_3g_6']
    df_fe['aug_vs_good_vbc_3g']   = df_fe['vbc_3g_8'] / (df_fe['vbc_3g_avg_678'] + 1)
    print("   ✔ vbc_3g trend features added (6→7, 7→8, overall, aug-vs-avg)")

   ✔ vbc_3g trend features added (6→7, 7→8, overall, aug-vs-avg)


In [26]:
# Roaming usage flag
df_fe['uses_roaming'] = ((df_fe['roam_ic_mou_6'] + df_fe['roam_ic_mou_7'] + df_fe['roam_ic_mou_8'] +
                          df_fe['roam_og_mou_6'] + df_fe['roam_og_mou_7'] + df_fe['roam_og_mou_8']) > 0).astype(int)

# Data user flag
df_fe['uses_data'] = ((df_fe['vol_2g_mb_6'] + df_fe['vol_2g_mb_7'] + df_fe['vol_2g_mb_8'] +
                       df_fe['vol_3g_mb_6'] + df_fe['vol_3g_mb_7'] + df_fe['vol_3g_mb_8']) > 0).astype(int)

# Total call ratio (OG vs IC)
total_og = df_fe[['total_og_mou_6', 'total_og_mou_7', 'total_og_mou_8']].sum(axis=1)
total_ic = df_fe[['total_ic_mou_6', 'total_ic_mou_7', 'total_ic_mou_8']].sum(axis=1)
df_fe['og_ic_ratio'] = total_og / (total_ic + 1)

# Aug vs Good phase (Jun+Jul) usage drop
df_fe['aug_vs_good_arpu']  = df_fe['arpu_8']          / (df_fe['arpu_avg_678']          + 1)
df_fe['aug_vs_good_og']    = df_fe['total_og_mou_8']  / (df_fe['total_og_mou_avg_678']  + 1)
df_fe['aug_vs_good_rech']  = df_fe['total_rech_amt_8']/ (df_fe['total_rech_amt_avg_678']+ 1)

In [27]:
new_feat_count = len([c for c in df_fe.columns if c in [
    'arpu_trend_6_to_7','arpu_trend_7_to_8','arpu_overall_trend',
    'og_trend_6_to_7','og_trend_7_to_8','og_overall_trend',
    'ic_trend_6_to_7','ic_trend_7_to_8','ic_overall_trend',
    'rech_trend_6_to_7','rech_trend_7_to_8','rech_overall_trend',
    'vbc_3g_trend_6_to_7','vbc_3g_trend_7_to_8','vbc_3g_overall_trend','aug_vs_good_vbc_3g',
    'uses_roaming','uses_data','og_ic_ratio',
    'aug_vs_good_arpu','aug_vs_good_og','aug_vs_good_rech'
]])

print(f"\n✅ Engineered {new_feat_count}+ new features:")
print(f"   • Revenue trends (ARPU decline signals)")
print(f"   • Call usage trends (OG/IC month-over-month)")
print(f"   • Recharge trends")
print(f"   • VBC-3G trends (voice-before-consume burn-rate signals)  ← NEW")
print(f"   • Average & std deviation across months 6-8 (incl. vbc_3g)  ← EXTENDED")
print(f"   • Behavioural flags (roaming, data usage)")
print(f"   • August vs 'good phase' ratio (early churn signal)")
print(f"\n   Total features after engineering: {df_fe.shape[1] - 1}")  # -1 for target


✅ Engineered 22+ new features:
   • Revenue trends (ARPU decline signals)
   • Call usage trends (OG/IC month-over-month)
   • Recharge trends
   • VBC-3G trends (voice-before-consume burn-rate signals)  ← NEW
   • Average & std deviation across months 6-8 (incl. vbc_3g)  ← EXTENDED
   • Behavioural flags (roaming, data usage)
   • August vs 'good phase' ratio (early churn signal)

   Total features after engineering: 196


In [28]:
# Handle missing values 
print(f"\n🔧 Handling missing values...")
miss_before = df_fe.drop(columns=['churn']).isnull().sum().sum()

# High missing columns: drop if >40%
high_miss = [c for c in df_fe.columns if df_fe[c].isnull().mean() > 0.4 and c != 'churn']
df_fe.drop(columns=high_miss, inplace=True)
print(f"   Dropped {len(high_miss)} columns with >40% missing")

# Remaining: fill with median
num_cols = df_fe.select_dtypes(include='number').columns.drop('churn')
for col in num_cols:
    if df_fe[col].isnull().any():
        df_fe[col] = df_fe[col].fillna(df_fe[col].median())

miss_after = df_fe.drop(columns=['churn']).isnull().sum().sum()
print(f"   Missing values: {miss_before:,} → {miss_after}")



🔧 Handling missing values...
   Dropped 27 columns with >40% missing
   Missing values: 2,391,506 → 0


In [29]:
# Feature correlation with churn 
X_cols = [c for c in df_fe.columns if c != 'churn']
corr_with_churn = df_fe[X_cols + ['churn']].corr()['churn'].drop('churn').abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_corr = corr_with_churn.head(30)
colors_bar = ['#e74c3c' if v > 0.1 else '#3498db' for v in top_corr.values]
ax.barh(range(len(top_corr)), top_corr.values, color=colors_bar, alpha=0.8)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index, fontsize=8)
ax.set_xlabel('|Correlation with Churn|')
ax.set_title('Section 3 — Feature Engineering: Top 30 Correlated Features with Churn',
             fontsize=11, fontweight='bold')
ax.axvline(0.1, color='orange', linestyle='--', label='0.10 threshold')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/3_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"\n✔ Top correlated features saved: 3_feature_correlation.png")
print(f"   Top 5 correlated features:")
for f, v in corr_with_churn.head(5).items():
    print(f"   → {f:40s}: {v:.4f}")


✔ Top correlated features saved: 3_feature_correlation.png
   Top 5 correlated features:
   → aug_vs_good_og                          : 0.3198
   → aug_vs_good_arpu                        : 0.3064
   → aug_vs_good_rech                        : 0.2931
   → arpu_overall_trend                      : 0.2192
   → rech_overall_trend                      : 0.2100


In [30]:

# SECTION 4 — MODEL BUILDING

print("\n" + "═"*65)
print("  SECTION 4: MODEL BUILDING")
print("═"*65)


═════════════════════════════════════════════════════════════════
  SECTION 4: MODEL BUILDING
═════════════════════════════════════════════════════════════════


In [31]:
# Prepare X and y
X = df_fe.drop(columns=['churn'])
y = df_fe['churn']

# Drop any remaining non-numeric columns
X = X.select_dtypes(include='number')

print(f"\n📐 Feature matrix: {X.shape[0]:,} samples × {X.shape[1]} features")
print(f"   Churn rate  : {y.mean()*100:.2f}%")


📐 Feature matrix: 99,999 samples × 169 features
   Churn rate  : 9.02%


In [32]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\n📊 Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"   Train churn rate: {y_train.mean()*100:.2f}%")
print(f"   Test  churn rate: {y_test.mean()*100:.2f}%")


📊 Train: 79,999 | Test: 20,000
   Train churn rate: 9.02%
   Test  churn rate: 9.02%


In [33]:
# Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Class weights (for imbalance)
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: cw[0], 1: cw[1]}
print(f"\n⚖  Class weights: {{0: {cw[0]:.2f}, 1: {cw[1]:.2f}}}")


⚖  Class weights: {0: 0.55, 1: 5.54}


In [34]:
# Results tracker 
results = {}

In [35]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, use_scaled=False):
    """Train, predict, and collect metrics."""
    Xtr = X_tr if not use_scaled else X_train_sc
    Xte = X_te if not use_scaled else X_test_sc

    t0 = time.time()
    model.fit(Xtr, y_tr)
    train_time = time.time() - t0

    y_pred      = model.predict(Xte)
    y_prob      = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    roc  = roc_auc_score(y_te, y_prob) if y_prob is not None else None
    ap   = average_precision_score(y_te, y_prob) if y_prob is not None else None

    results[name] = {
        'model': model, 'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1': f1, 'roc_auc': roc, 'avg_precision': ap,
        'train_time': train_time, 'y_pred': y_pred, 'y_prob': y_prob
    }

    print(f"\n   {'─'*55}")
    print(f"   Model : {name}")
    print(f"   {'─'*55}")
    print(f"   Accuracy      : {acc:.4f}")
    print(f"   Precision     : {prec:.4f}")
    print(f"   Recall        : {rec:.4f}")
    print(f"   F1-Score      : {f1:.4f}")
    if roc: print(f"   ROC-AUC       : {roc:.4f}")
    if ap:  print(f"   Avg Precision : {ap:.4f}")
    print(f"   Train time    : {train_time:.2f}s")
    return model


In [36]:

# MODEL 1: Logistic Regression (Baseline)

print("\n" + "─"*65)
print("  MODEL 1: LOGISTIC REGRESSION (Baseline)")
print("─"*65)


─────────────────────────────────────────────────────────────────
  MODEL 1: LOGISTIC REGRESSION (Baseline)
─────────────────────────────────────────────────────────────────


In [37]:
lr = LogisticRegression(
    class_weight='balanced', max_iter=1000,
    solver='saga', random_state=RANDOM_STATE, n_jobs=-1
)
evaluate_model('Logistic Regression', lr, X_train, X_test, y_train, y_test, use_scaled=True)



   ───────────────────────────────────────────────────────
   Model : Logistic Regression
   ───────────────────────────────────────────────────────
   Accuracy      : 0.8709
   Precision     : 0.3980
   Recall        : 0.8409
   F1-Score      : 0.5402
   ROC-AUC       : 0.9245
   Avg Precision : 0.6367
   Train time    : 257.10s


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [39]:

# MODEL 2: Naive Bayes 

print("\n" + "─"*65)
print("  MODEL 2: NAIVE BAYES")
print("─"*65)


─────────────────────────────────────────────────────────────────
  MODEL 2: NAIVE BAYES
─────────────────────────────────────────────────────────────────


In [40]:
nb = GaussianNB()
evaluate_model('Naive Bayes', nb, X_train, X_test, y_train, y_test, use_scaled=True)


   ───────────────────────────────────────────────────────
   Model : Naive Bayes
   ───────────────────────────────────────────────────────
   Accuracy      : 0.4960
   Precision     : 0.1406
   Recall        : 0.8975
   F1-Score      : 0.2431
   ROC-AUC       : 0.8153
   Avg Precision : 0.2965
   Train time    : 0.23s


,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [41]:

# MODEL 3: Decision Tree

print("\n" + "─"*65)
print("  MODEL 3: DECISION TREE")
print("─"*65)



─────────────────────────────────────────────────────────────────
  MODEL 3: DECISION TREE
─────────────────────────────────────────────────────────────────


In [42]:
dt = DecisionTreeClassifier(
    class_weight='balanced', max_depth=8, min_samples_leaf=50,
    random_state=RANDOM_STATE
)
evaluate_model('Decision Tree', dt, X_train, X_test, y_train, y_test)


   ───────────────────────────────────────────────────────
   Model : Decision Tree
   ───────────────────────────────────────────────────────
   Accuracy      : 0.8800
   Precision     : 0.4182
   Recall        : 0.8448
   F1-Score      : 0.5595
   ROC-AUC       : 0.9209
   Avg Precision : 0.7056
   Train time    : 5.00s


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

In [43]:

# MODEL 4: K-Nearest Neighbours 

print("\n" + "─"*65)
print("  MODEL 4: K-NEAREST NEIGHBOURS")
print("─"*65)

knn = KNeighborsClassifier(n_neighbors=11, n_jobs=-1)
evaluate_model('KNN', knn, X_train, X_test, y_train, y_test, use_scaled=True)


─────────────────────────────────────────────────────────────────
  MODEL 4: K-NEAREST NEIGHBOURS
─────────────────────────────────────────────────────────────────

   ───────────────────────────────────────────────────────
   Model : KNN
   ───────────────────────────────────────────────────────
   Accuracy      : 0.9293
   Precision     : 0.6560
   Recall        : 0.4545
   F1-Score      : 0.5370
   ROC-AUC       : 0.8799
   Avg Precision : 0.5543
   Train time    : 0.06s


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",-1


In [44]:

# MODEL 5: Random Forest 

print("\n" + "─"*65)
print("  MODEL 5: RANDOM FOREST")
print("─"*65)

rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    max_depth=15, min_samples_leaf=20,
    max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE
)
evaluate_model('Random Forest', rf, X_train, X_test, y_train, y_test)


─────────────────────────────────────────────────────────────────
  MODEL 5: RANDOM FOREST
─────────────────────────────────────────────────────────────────

   ───────────────────────────────────────────────────────
   Model : Random Forest
   ───────────────────────────────────────────────────────
   Accuracy      : 0.9321
   Precision     : 0.5972
   Recall        : 0.7594
   F1-Score      : 0.6686
   ROC-AUC       : 0.9404
   Avg Precision : 0.7362
   Train time    : 31.66s


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

In [45]:

# MODEL 6: Gradient Boosting 

print("\n" + "─"*65)
print("  MODEL 6: GRADIENT BOOSTING")
print("─"*65)

gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05,
    max_depth=5, subsample=0.8,
    min_samples_leaf=30, random_state=RANDOM_STATE
)
# Use class weights via sample_weight
sample_weights = np.where(y_train == 1, cw[1], cw[0])
gb.fit(X_train, y_train, sample_weight=sample_weights)

y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

results['Gradient Boosting'] = {
    'model': gb,
    'accuracy': accuracy_score(y_test, y_pred_gb),
    'precision': precision_score(y_test, y_pred_gb, zero_division=0),
    'recall': recall_score(y_test, y_pred_gb, zero_division=0),
    'f1': f1_score(y_test, y_pred_gb, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob_gb),
    'avg_precision': average_precision_score(y_test, y_prob_gb),
    'train_time': 0, 'y_pred': y_pred_gb, 'y_prob': y_prob_gb
}

r = results['Gradient Boosting']
print(f"\n   {'─'*55}")
print(f"   Model : Gradient Boosting")
print(f"   {'─'*55}")
print(f"   Accuracy      : {r['accuracy']:.4f}")
print(f"   Precision     : {r['precision']:.4f}")
print(f"   Recall        : {r['recall']:.4f}")
print(f"   F1-Score      : {r['f1']:.4f}")
print(f"   ROC-AUC       : {r['roc_auc']:.4f}")
print(f"   Avg Precision : {r['avg_precision']:.4f}")


─────────────────────────────────────────────────────────────────
  MODEL 6: GRADIENT BOOSTING
─────────────────────────────────────────────────────────────────

   ───────────────────────────────────────────────────────
   Model : Gradient Boosting
   ───────────────────────────────────────────────────────
   Accuracy      : 0.9086
   Precision     : 0.4961
   Recall        : 0.8492
   F1-Score      : 0.6263
   ROC-AUC       : 0.9461
   Avg Precision : 0.7335


In [46]:
# MODEL 6b: LightGBM 
print("\n" + "─"*65)
print("  MODEL 6b: LIGHTGBM")
print("─"*65)
lgbm = LGBMClassifier(
    n_estimators=300, learning_rate=0.05,
    max_depth=6, num_leaves=31, subsample=0.8,
    colsample_bytree=0.8, min_child_samples=30,
    class_weight='balanced', random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1
)
evaluate_model('LightGBM', lgbm, X_train, X_test, y_train, y_test)


─────────────────────────────────────────────────────────────────
  MODEL 6b: LIGHTGBM
─────────────────────────────────────────────────────────────────

   ───────────────────────────────────────────────────────
   Model : LightGBM
   ───────────────────────────────────────────────────────
   Accuracy      : 0.9120
   Precision     : 0.5072
   Recall        : 0.8409
   F1-Score      : 0.6327
   ROC-AUC       : 0.9454
   Avg Precision : 0.7461
   Train time    : 3.55s


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,30


In [47]:

# MODEL 7: Random Forest + Hyperparameter Tuning 

print("\n" + "─"*65)
print("  MODEL 7: RANDOM FOREST — HYPERPARAMETER TUNING (RandomizedSearchCV)")
print("─"*65)

param_grid_rf = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [10, 15, 20, None],
    'min_samples_leaf': [10, 20, 50],
    'max_features'    : ['sqrt', 'log2'],
    'class_weight'    : ['balanced', 'balanced_subsample'],
}

rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rscv_rf = RandomizedSearchCV(
    rf_base, param_grid_rf, n_iter=12,
    scoring='roc_auc', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
rscv_rf.fit(X_train, y_train)

best_rf = rscv_rf.best_estimator_
print(f"\n   Best params: {rscv_rf.best_params_}")
print(f"   Best CV ROC-AUC: {rscv_rf.best_score_:.4f}")
evaluate_model('RF Tuned', best_rf, X_train, X_test, y_train, y_test)


─────────────────────────────────────────────────────────────────
  MODEL 7: RANDOM FOREST — HYPERPARAMETER TUNING (RandomizedSearchCV)
─────────────────────────────────────────────────────────────────

   Best params: {'n_estimators': 200, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_depth': 15, 'class_weight': 'balanced'}
   Best CV ROC-AUC: 0.9387

   ───────────────────────────────────────────────────────
   Model : RF Tuned
   ───────────────────────────────────────────────────────
   Accuracy      : 0.9391
   Precision     : 0.6426
   Recall        : 0.7317
   F1-Score      : 0.6843
   ROC-AUC       : 0.9401
   Avg Precision : 0.7372
   Train time    : 45.14s


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

In [48]:

# MODEL 8: Gradient Boosting — Hyperparameter Tuning 

print("\n" + "─"*65)
print("  MODEL 8: GRADIENT BOOSTING — HYPERPARAMETER TUNING")
print("─"*65)

param_grid_gb = {
    'n_estimators'    : [100, 200, 300],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'max_depth'       : [3, 5, 7],
    'subsample'       : [0.7, 0.8, 1.0],
    'min_samples_leaf': [20, 50],
}

gb_base = GradientBoostingClassifier(random_state=RANDOM_STATE)
rscv_gb = RandomizedSearchCV(
    gb_base, param_grid_gb, n_iter=10,
    scoring='roc_auc', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)

sample_weights_all = np.where(y_train == 1, cw[1], cw[0])
rscv_gb.fit(X_train, y_train, sample_weight=sample_weights_all)

best_gb = rscv_gb.best_estimator_
print(f"\n   Best params: {rscv_gb.best_params_}")
print(f"   Best CV ROC-AUC: {rscv_gb.best_score_:.4f}")

best_gb.fit(X_train, y_train, sample_weight=sample_weights_all)
y_pred_bst = best_gb.predict(X_test)
y_prob_bst = best_gb.predict_proba(X_test)[:, 1]

results['GB Tuned'] = {
    'model': best_gb,
    'accuracy': accuracy_score(y_test, y_pred_bst),
    'precision': precision_score(y_test, y_pred_bst, zero_division=0),
    'recall': recall_score(y_test, y_pred_bst, zero_division=0),
    'f1': f1_score(y_test, y_pred_bst, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob_bst),
    'avg_precision': average_precision_score(y_test, y_prob_bst),
    'train_time': 0, 'y_pred': y_pred_bst, 'y_prob': y_prob_bst
}

r = results['GB Tuned']
print(f"\n   Accuracy      : {r['accuracy']:.4f}")
print(f"   Precision     : {r['precision']:.4f}")
print(f"   Recall        : {r['recall']:.4f}")
print(f"   F1-Score      : {r['f1']:.4f}")
print(f"   ROC-AUC       : {r['roc_auc']:.4f}")
print(f"   Avg Precision : {r['avg_precision']:.4f}")


─────────────────────────────────────────────────────────────────
  MODEL 8: GRADIENT BOOSTING — HYPERPARAMETER TUNING
─────────────────────────────────────────────────────────────────

   Best params: {'subsample': 0.7, 'n_estimators': 300, 'min_samples_leaf': 50, 'max_depth': 7, 'learning_rate': 0.01}
   Best CV ROC-AUC: 0.9432

   Accuracy      : 0.9120
   Precision     : 0.5073
   Recall        : 0.8420
   F1-Score      : 0.6332
   ROC-AUC       : 0.9458
   Avg Precision : 0.7416


In [49]:
# Model 8b: LightGBM — Hyperparameter Tuning
print("\n" + "─"*65)
print("  MODEL 6c: LIGHTGBM — HYPERPARAMETER TUNING")
print("─"*65)
param_grid_lgbm = {
    'n_estimators'     : [100, 200, 300],
    'learning_rate'    : [0.01, 0.05, 0.1],
    'max_depth'        : [4, 6, 8, -1],
    'num_leaves'       : [15, 31, 63],
    'subsample'        : [0.7, 0.8, 1.0],
    'colsample_bytree' : [0.7, 0.8, 1.0],
    'min_child_samples': [20, 30, 50],
}

lgbm_base = LGBMClassifier(
    class_weight='balanced', random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1
)

rscv_lgbm = RandomizedSearchCV(
    lgbm_base, param_grid_lgbm, n_iter=12,
    scoring='roc_auc', cv=cv_strat,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
rscv_lgbm.fit(X_train, y_train)

best_lgbm = rscv_lgbm.best_estimator_
print(f"\n   Best params: {rscv_lgbm.best_params_}")
print(f"   Best CV ROC-AUC: {rscv_lgbm.best_score_:.4f}")
evaluate_model('LGBM Tuned', best_lgbm, X_train, X_test, y_train, y_test)



─────────────────────────────────────────────────────────────────
  MODEL 6c: LIGHTGBM — HYPERPARAMETER TUNING
─────────────────────────────────────────────────────────────────

   Best params: {'subsample': 0.8, 'num_leaves': 63, 'n_estimators': 100, 'min_child_samples': 50, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
   Best CV ROC-AUC: 0.9432

   ───────────────────────────────────────────────────────
   Model : LGBM Tuned
   ───────────────────────────────────────────────────────
   Accuracy      : 0.9070
   Precision     : 0.4909
   Recall        : 0.8487
   F1-Score      : 0.6220
   ROC-AUC       : 0.9447
   Avg Precision : 0.7436
   Train time    : 6.95s


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,6
,learning_rate,0.05
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,50


In [50]:

# MODEL 9: Voting Ensemble (Best Models Combined)
print("\n" + "─"*65)
print("  MODEL 9: VOTING ENSEMBLE (RF Tuned + GB Tuned + LR)")
print("─"*65)

voting = VotingClassifier(
    estimators=[
        ('rf', best_rf),
        ('gb', best_gb),
        ('lgbm', best_lgbm),
        ('lr', lr)
    ],
    voting='soft',
    n_jobs=-1
)
evaluate_model('Voting Ensemble', voting, X_train, X_test, y_train, y_test)


─────────────────────────────────────────────────────────────────
  MODEL 9: VOTING ENSEMBLE (RF Tuned + GB Tuned + LR)
─────────────────────────────────────────────────────────────────

   ───────────────────────────────────────────────────────
   Model : Voting Ensemble
   ───────────────────────────────────────────────────────
   Accuracy      : 0.9366
   Precision     : 0.6222
   Recall        : 0.7578
   F1-Score      : 0.6833
   ROC-AUC       : 0.9439
   Avg Precision : 0.7435
   Train time    : 1711.68s


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('rf', ...), ('gb', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",-1
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10


In [51]:

# MODEL COMPARISON & VISUALIZATIONS

print("\n" + "═"*65)
print("  SECTION 5: MODEL COMPARISON & RESULTS")
print("═"*65)


═════════════════════════════════════════════════════════════════
  SECTION 5: MODEL COMPARISON & RESULTS
═════════════════════════════════════════════════════════════════


In [52]:
# Comparison table 
metrics_df = pd.DataFrame({
    name: {
        'Accuracy'     : r['accuracy'],
        'Precision'    : r['precision'],
        'Recall'       : r['recall'],
        'F1-Score'     : r['f1'],
        'ROC-AUC'      : r['roc_auc'] if r['roc_auc'] else 0,
        'Avg Precision': r['avg_precision'] if r['avg_precision'] else 0,
    }
    for name, r in results.items()
}).T.round(4)

print("\n📊 Model Comparison Table:")
print(metrics_df.to_string())

best_model_name = metrics_df['ROC-AUC'].idxmax()
print(f"\n🏆 Best Model by ROC-AUC: {best_model_name}")
print(f"   ROC-AUC   : {metrics_df.loc[best_model_name,'ROC-AUC']:.4f}")
print(f"   F1-Score  : {metrics_df.loc[best_model_name,'F1-Score']:.4f}")
print(f"   Recall    : {metrics_df.loc[best_model_name,'Recall']:.4f}")



📊 Model Comparison Table:
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC  Avg Precision
Logistic Regression    0.8709     0.3980  0.8409    0.5402   0.9245         0.6367
Naive Bayes            0.4960     0.1406  0.8975    0.2431   0.8153         0.2965
Decision Tree          0.8800     0.4182  0.8448    0.5595   0.9209         0.7056
KNN                    0.9293     0.6560  0.4545    0.5370   0.8799         0.5543
Random Forest          0.9321     0.5972  0.7594    0.6686   0.9404         0.7362
Gradient Boosting      0.9086     0.4961  0.8492    0.6263   0.9461         0.7335
LightGBM               0.9120     0.5072  0.8409    0.6327   0.9454         0.7461
RF Tuned               0.9391     0.6426  0.7317    0.6843   0.9401         0.7372
GB Tuned               0.9120     0.5073  0.8420    0.6332   0.9458         0.7416
LGBM Tuned             0.9070     0.4909  0.8487    0.6220   0.9447         0.7436
Voting Ensemble        0.9366     0.6222  0.7578    0.6833  

In [53]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Section 5 — Model Comparison Dashboard', fontsize=15, fontweight='bold')

model_names = list(metrics_df.index)          # ← derive from metrics_df, not results
palette = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

for ax, metric in zip(axes.flat, ['ROC-AUC', 'F1-Score', 'Recall', 'Precision']):
    vals = metrics_df[metric].values
    bars = ax.bar(range(len(model_names)), vals, color=palette, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=35, ha='right', fontsize=8)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, min(vals.max() + 0.1, 1.0))
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)
    # Highlight best
    best_idx = np.argmax(vals)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/5a_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()


In [54]:
# 5c. ROC curves 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Section 5 — ROC & Precision-Recall Curves', fontsize=13, fontweight='bold')

colors_roc = plt.cm.tab10(np.linspace(0, 1, len(results)))
for (name, r), col in zip(results.items(), colors_roc):
    if r['y_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
        axes[0].plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})", color=col, linewidth=1.5)

        prec_c, rec_c, _ = precision_recall_curve(y_test, r['y_prob'])
        axes[1].plot(rec_c, prec_c, label=f"{name} (AP={r['avg_precision']:.3f})", color=col, linewidth=1.5)

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(fontsize=7, loc='lower right')
axes[0].grid(True, alpha=0.3)

axes[1].axhline(y=y_test.mean(), color='black', linestyle='--', label=f'Baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(fontsize=7, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/5b_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.close()

In [55]:
# Confusion matrices for top 3 models 
top3 = metrics_df['ROC-AUC'].nlargest(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Section 5 — Confusion Matrices (Top 3 Models)', fontsize=13, fontweight='bold')

for ax, name in zip(axes, top3):
    r = results[name]
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Churn', 'Churn'],
                yticklabels=['Not Churn', 'Churn'])
    ax.set_title(f'{name}\n(F1={r["f1"]:.3f}, AUC={r["roc_auc"]:.3f})', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/5c_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()


In [56]:
# Feature Importance (Best Tree Model) 
best_tree_name = metrics_df.loc[
    ['Random Forest', 'RF Tuned', 'Gradient Boosting', 'GB Tuned', 'LightGBM', 'LGBM Tuned'],
    'ROC-AUC'].idxmax()
best_tree_model = results[best_tree_name]['model']

feat_imp = pd.Series(best_tree_model.feature_importances_, index=X.columns).sort_values(ascending=False)
top_feat = feat_imp.head(25)

fig, ax = plt.subplots(figsize=(10, 9))
colors_fi = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(top_feat))]
ax.barh(range(len(top_feat)), top_feat.values[::-1], color=colors_fi[::-1], alpha=0.85)
ax.set_yticks(range(len(top_feat)))
ax.set_yticklabels(top_feat.index[::-1], fontsize=9)
ax.set_xlabel('Feature Importance')
ax.set_title(f'Section 5 — Top 25 Features ({best_tree_name})', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/5d_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()

print("\n✔ All plots saved.")


✔ All plots saved.


In [57]:
# Best model classification report
print(f"\n📋 Classification Report — Best Model ({best_model_name}):")
best_r = results[best_model_name]
print(classification_report(y_test, best_r['y_pred'], target_names=['Not Churn', 'Churn']))



📋 Classification Report — Best Model (Gradient Boosting):
              precision    recall  f1-score   support

   Not Churn       0.98      0.91      0.95     18196
       Churn       0.50      0.85      0.63      1804

    accuracy                           0.91     20000
   macro avg       0.74      0.88      0.79     20000
weighted avg       0.94      0.91      0.92     20000



In [58]:
# Threshold analysis
print("\n🔍 Threshold Analysis (Best Model):")
if best_r['y_prob'] is not None:
    thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
    print(f"   {'Threshold':>12} {'Precision':>10} {'Recall':>10} {'F1':>8} {'Flagged':>10}")
    print(f"   {'-'*55}")
    for thr in thresholds:
        y_pred_thr = (best_r['y_prob'] >= thr).astype(int)
        p = precision_score(y_test, y_pred_thr, zero_division=0)
        r = recall_score(y_test, y_pred_thr, zero_division=0)
        f = f1_score(y_test, y_pred_thr, zero_division=0)
        flagged = y_pred_thr.sum()
        print(f"   {thr:>12.2f} {p:>10.4f} {r:>10.4f} {f:>8.4f} {flagged:>10,}")


🔍 Threshold Analysis (Best Model):
      Threshold  Precision     Recall       F1    Flagged
   -------------------------------------------------------
           0.30     0.3773     0.9008   0.5318      4,307
           0.40     0.4441     0.8742   0.5890      3,551
           0.50     0.4961     0.8492   0.6263      3,088
           0.60     0.5531     0.8176   0.6598      2,667
           0.70     0.6108     0.7683   0.6806      2,269


In [59]:
# Cross-validation final check 
print(f"\n🔄 Cross-Validation (5-fold) — Best Model ({best_model_name}):")
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(best_r['model'], X_train, y_train, cv=cv5, scoring='roc_auc', n_jobs=-1)
print(f"   CV ROC-AUC scores : {cv_scores.round(4)}")
print(f"   Mean ± Std        : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


🔄 Cross-Validation (5-fold) — Best Model (Gradient Boosting):
   CV ROC-AUC scores : [0.9491 0.942  0.9445 0.9432 0.935 ]
   Mean ± Std        : 0.9428 ± 0.0046


In [71]:
# Summary chart 
fig, ax = plt.subplots(figsize=(14, 5))
model_progression = ['Logistic Regression', 'Naive Bayes', 'Decision Tree', 'KNN',
                     'Random Forest', 'Gradient Boosting','LightGBM', 'RF Tuned', 'GB Tuned', 'LGBM Tuned', 'Voting Ensemble']
model_progression = [m for m in model_progression if m in results]
auc_progression   = [results[m]['roc_auc'] for m in model_progression]
f1_progression    = [results[m]['f1'] for m in model_progression]

x = range(len(model_progression))
ax.plot(x, auc_progression, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='ROC-AUC')
ax.plot(x, f1_progression,  's-', color='#3498db', linewidth=2, markersize=8, label='F1-Score')
for i, (auc, f1) in enumerate(zip(auc_progression, f1_progression)):
    ax.annotate(f'{auc:.3f}', (i, auc), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8, color='#e74c3c')
    ax.annotate(f'{f1:.3f}',  (i, f1),  textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8, color='#3498db')

ax.set_xticks(x)
ax.set_xticklabels(model_progression, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Model Progression — From Baseline to Best Model', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/5e_model_progression.png', dpi=150, bbox_inches='tight')
plt.close()


In [61]:
# SECTION 6 — CHURN-FLAG SCORING & CAMPAIGN EXPORT
print("\n" + "═"*65)
print("  SECTION 6: CHURN-FLAG SCORING & CAMPAIGN EXPORT")
print("═"*65)


═════════════════════════════════════════════════════════════════
  SECTION 6: CHURN-FLAG SCORING & CAMPAIGN EXPORT
═════════════════════════════════════════════════════════════════


In [62]:
from sklearn.base import clone

CHURN_THRESHOLD = 0.45   # chosen for email campaigns: favors recall (catch more
                          # at-risk customers) since a false-positive email costs
                          # little, but a missed churner costs the customer entirely

# Models trained on scaled features need scaled input again at scoring time
SCALED_MODELS = {'Logistic Regression', 'Naive Bayes', 'KNN'}

In [63]:
# ── 6a. Refit best model on the FULL dataset ──────────────────
# Note: best_r['model'] was only trained on the 80% train split.
# For production scoring we retrain an identical model on all
# available data so it uses every customer's signal.
print(f"\n🔁 Refitting best model ({best_model_name}) on full dataset...")

final_model = clone(best_r['model'])

if best_model_name in SCALED_MODELS:
    scaler_full = StandardScaler()
    X_full_sc = scaler_full.fit_transform(X)
    final_model.fit(X_full_sc, y)
    churn_prob = final_model.predict_proba(X_full_sc)[:, 1]
else:
    # Gradient Boosting / GB Tuned need sample_weight for class balance;
    # tree ensembles with class_weight='balanced' handle it internally
    if 'class_weight' not in final_model.get_params():
        cw_full = compute_class_weight('balanced', classes=np.array([0, 1]), y=y)
        sw_full = np.where(y == 1, cw_full[1], cw_full[0])
        final_model.fit(X, y, sample_weight=sw_full)
    else:
        final_model.fit(X, y)
    churn_prob = final_model.predict_proba(X)[:, 1]

print(f"   ✔ Model refit on {X.shape[0]:,} customers")


🔁 Refitting best model (Gradient Boosting) on full dataset...
   ✔ Model refit on 99,999 customers


In [64]:
# ── 6b. Build CHURN_FLAG and risk tiers ────────────────────────
churn_flag = (churn_prob >= CHURN_THRESHOLD).astype(int)

def risk_tier(p):
    if p >= 0.75:
        return 'Critical'
    elif p >= CHURN_THRESHOLD:
        return 'High'
    elif p >= 0.25:
        return 'Medium'
    else:
        return 'Low'

risk_tiers = [risk_tier(p) for p in churn_prob]

In [65]:
# ── 6c. Reattach customer identifiers ──────────────────────────
# mobile_number was dropped during feature engineering (Section 3b),
# so pull it back from the original df using matching row index.
scored_df = pd.DataFrame({
    'mobile_number'      : df.loc[X.index, 'mobile_number'].values,
    'churn_probability'  : churn_prob.round(4),
    'CHURN_FLAG'         : churn_flag,
    'CHURN_FLAG_label'   : np.where(churn_flag == 1, 'YES', 'NO'),
    'risk_tier'          : risk_tiers
}).sort_values('churn_probability', ascending=False).reset_index(drop=True)

In [66]:
# ── 6d. Summary stats ───────────────────────────────────────────
print(f"\n📊 CHURN-FLAG Summary (threshold = {CHURN_THRESHOLD}):")
print(f"   Total customers scored : {len(scored_df):,}")
print(f"   CHURN_FLAG = YES       : {(scored_df['CHURN_FLAG']==1).sum():,} "
      f"({(scored_df['CHURN_FLAG']==1).mean()*100:.1f}%)")
print(f"   CHURN_FLAG = NO        : {(scored_df['CHURN_FLAG']==0).sum():,} "
      f"({(scored_df['CHURN_FLAG']==0).mean()*100:.1f}%)")

print(f"\n📊 Risk Tier Breakdown:")
tier_counts = scored_df['risk_tier'].value_counts().reindex(['Critical', 'High', 'Medium', 'Low'])
for tier, count in tier_counts.items():
    print(f"   {tier:<10s}: {count:>7,} customers  ({count/len(scored_df)*100:.1f}%)")



📊 CHURN-FLAG Summary (threshold = 0.45):
   Total customers scored : 99,999
   CHURN_FLAG = YES       : 17,362 (17.4%)
   CHURN_FLAG = NO        : 82,637 (82.6%)

📊 Risk Tier Breakdown:
   Critical  :  11,013 customers  (11.0%)
   High      :   6,349 customers  (6.3%)
   Medium    :   7,220 customers  (7.2%)
   Low       :  75,417 customers  (75.4%)


In [72]:
# ── 6e. Risk tier visualization ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section 6 — Customer Risk Segmentation for Campaign Targeting', fontsize=13, fontweight='bold')

tier_colors = {'Critical': '#c0392b', 'High': '#e67e22', 'Medium': '#f1c40f', 'Low': '#2ecc71'}
axes[0].bar(tier_counts.index, tier_counts.values,
            color=[tier_colors[t] for t in tier_counts.index], alpha=0.85, edgecolor='black')
axes[0].set_title('Customers by Risk Tier')
axes[0].set_ylabel('Number of Customers')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v + len(scored_df)*0.005, f'{v:,}', ha='center', fontsize=9)

axes[1].hist(scored_df['churn_probability'], bins=50, color='#3498db', alpha=0.8, edgecolor='black', linewidth=0.3)
axes[1].axvline(CHURN_THRESHOLD, color='red', linestyle='--', linewidth=2,
                 label=f'CHURN_FLAG cutoff ({CHURN_THRESHOLD})')
axes[1].set_title('Churn Probability Distribution (All Customers)')
axes[1].set_xlabel('Predicted Churn Probability')
axes[1].set_ylabel('Number of Customers')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/6_risk_segmentation.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✔ Plot saved: 6_risk_segmentation.png")


✔ Plot saved: 6_risk_segmentation.png


In [73]:
# ── 6f. Export campaign-ready CSV ──────────────────────────────
export_path = f'{OUTPUT_DIR}/churn_flag_customer_scores.csv'
scored_df.to_csv(export_path, index=False)
print(f"\n✔ Exported: churn_flag_customer_scores.csv  ({len(scored_df):,} rows)")
print(f"   Columns: mobile_number, churn_probability, CHURN_FLAG, CHURN_FLAG_label, risk_tier")
print(f"\n   → Marketing: filter CHURN_FLAG_label == 'YES' for retention email campaigns")
print(f"   → Support  : filter risk_tier in ['Critical', 'High'] to auto-prioritize tickets")


✔ Exported: churn_flag_customer_scores.csv  (99,999 rows)
   Columns: mobile_number, churn_probability, CHURN_FLAG, CHURN_FLAG_label, risk_tier

   → Marketing: filter CHURN_FLAG_label == 'YES' for retention email campaigns
   → Support  : filter risk_tier in ['Critical', 'High'] to auto-prioritize tickets


In [74]:
# FINAL SUMMARY

print("\n" + "═"*65)
print("  FINAL SUMMARY")
print("═"*65)

print(f"""
  Dataset       : 99,999 customers × 226 raw features
  Target        : Churn in September (9.02% churn rate)
  Final Features: {X.shape[1]} (after engineering + selection)
  Train/Test    : 80/20 split, stratified

  ┌─────────────────────────────────────────────────────────┐
  │              MODEL LEADERBOARD (by ROC-AUC)             │
  ├──────────────────────┬──────────┬──────────┬────────────┤
  │ Model                │ ROC-AUC  │ F1-Score │ Recall     │
  ├──────────────────────┼──────────┼──────────┼────────────┤""")

for name in metrics_df['ROC-AUC'].sort_values(ascending=False).index:
    r = metrics_df.loc[name]
    marker = ' ← BEST' if name == best_model_name else ''
    print(f"  │ {name:<20s} │  {r['ROC-AUC']:.4f}  │  {r['F1-Score']:.4f}  │  {r['Recall']:.4f}    │{marker}")

print(f"""  └──────────────────────┴──────────┴──────────┴────────────┘

  🏆 Best Model     : {best_model_name}
     ROC-AUC        : {metrics_df.loc[best_model_name,'ROC-AUC']:.4f}
     CV Mean AUC    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}

  📁 Output files saved to /mnt/user-data/outputs/
     1_data_overview.png
     2a_usage_trends.png
     2b_arpu_distribution.png
     2c_churn_behaviour.png
     2d_correlation_heatmap.png
     3_feature_correlation.png
     5a_model_comparison.png
     5b_roc_pr_curves.png
     5c_confusion_matrices.png
     5d_feature_importance.png
     5e_model_progression.png
""")
print("✔ All done!")



═════════════════════════════════════════════════════════════════
  FINAL SUMMARY
═════════════════════════════════════════════════════════════════

  Dataset       : 99,999 customers × 226 raw features
  Target        : Churn in September (9.02% churn rate)
  Final Features: 169 (after engineering + selection)
  Train/Test    : 80/20 split, stratified

  ┌─────────────────────────────────────────────────────────┐
  │              MODEL LEADERBOARD (by ROC-AUC)             │
  ├──────────────────────┬──────────┬──────────┬────────────┤
  │ Model                │ ROC-AUC  │ F1-Score │ Recall     │
  ├──────────────────────┼──────────┼──────────┼────────────┤
  │ Gradient Boosting    │  0.9461  │  0.6263  │  0.8492    │ ← BEST
  │ GB Tuned             │  0.9458  │  0.6332  │  0.8420    │
  │ LightGBM             │  0.9454  │  0.6327  │  0.8409    │
  │ LGBM Tuned           │  0.9447  │  0.6220  │  0.8487    │
  │ Voting Ensemble      │  0.9439  │  0.6833  │  0.7578    │
  │ Random Fores

## Target Goals — Achievement Summary


| **Goal**                                                          | **Status** | **How It's Delivered**                                                                                                                                                         |
| ----------------------------------------------------------------- | ---------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **1. Understand variables influencing migration**                 | ✅ Achieved | Feature correlation ranking (Section 3e), feature importance (Section 5e), and EDA analysis of usage, recharge, and tenure trends (Section 2).                                 |
| **2. Churn risk scores for retention campaigns**                  | ✅ Achieved | Generated a **`churn_probability`** score for each customer across the full **99,999-customer** dataset and exported the results to **`churn_flag_customer_scores.csv`**.      |
| **3. CHURN-FLAG (YES/NO) for targeted email campaigns**           | ✅ Achieved | Created a binary **`CHURN_FLAG` / `CHURN_FLAG_label`** column using a business-defined threshold of **0.45**, enabling Marketing teams to directly filter high-risk customers. |
| **Bonus: Customer touchpoint prioritization (Support & Tickets)** | ✅ Achieved | Implemented a **`risk_tier`** column (**Critical / High / Medium / Low**) to automatically prioritize customer support tickets and retention efforts based on churn risk.      |


## Conclusion

This project delivered a complete, end-to-end machine learning pipeline to help No-Churn identify and act on high-value customer attrition before it happens. Starting from 226 raw features across 99,999 customers spanning four months (June–September), the pipeline cleaned, engineered, and reduced the feature space to 169 predictive signals, then benchmarked nine models spanning linear, distance-based, tree-based, and ensemble families.
Gradient Boosting emerged as the best-performing model, achieving a ROC-AUC of 0.9461 and recall of 84.9% on held-out data — meaning it correctly identifies roughly 85 out of every 100 customers who are about to churn. Beyond model benchmarking, the pipeline was extended into a full business deliverable: the winning model was refit on the entire customer base, scored to produce a per-customer churn probability, thresholded at 0.45 into a binary CHURN_FLAG (YES/NO), and further segmented into risk tiers (Critical/High/Medium/Low) — all exported to a campaign-ready CSV that Marketing and Support teams can directly filter and act on.
The result is not just a validated model, but an operational artifact: a scored customer list that closes the loop between "we can predict churn" and "here is exactly who to target and how urgently."

## Key Achievements

**Strong predictive performance**: 
Best model achieved 0.9461 ROC-AUC and 84.9% recall, confirmed stable via 5-fold cross-validation (not just a lucky train/test split).

**Rigorous leakage prevention**: 
All month-9 columns (the churn-defining period) were explicitly identified and dropped before training, with a data-quality fix (renaming jun/jul/aug/sep_vbc_3g to numeric-month convention) ensuring these features weren't silently mishandled in leakage removal or trend engineering.

**Business-aligned feature engineering**: 
Trend and ratio features (ARPU decline, recharge drop-off, "August vs. good-phase" usage ratios) captured early warning signals of disengagement — directly answering Goal 1 (understanding what drives migration).
Correctly handled severe class imbalance (9.02% churn rate) via class-weighting and stratified sampling, prioritizing recall over raw accuracy — the right tradeoff for a use case where missing a churner is far costlier than a false alarm.

**Comprehensive model comparison**: 
Nine models benchmarked identically, with hyperparameter tuning (RandomizedSearchCV) applied to the top tree-based candidates for a fair "best effort" comparison.

**Churn risk scoring delivered end-to-end (Goal 2)**: 
churn_probability computed for every customer in the full dataset, not just the test split — directly usable to prioritize retention effort.

**CHURN_FLAG variable delivered (Goal 3)**:
Binary YES/NO flag at a deliberately chosen threshold (0.45, tuned for low-cost email campaigns that favor catching more true churners over minimizing false positives), exported as churn_flag_customer_scores.csv.

**Support-touchpoint prioritization enabled**: 
Risk tiers (Critical/High/Medium/Low) allow support and ticketing systems to auto-escalate high-risk customers, extending the model's value beyond marketing into customer care.

**Reproducible, well-documented pipeline**: 
Single script covering EDA, feature engineering, model training, tuning, evaluation, and business export, with 12 saved diagnostic visualizations and clear inline rationale for every major decision (threshold choice, class-weighting method, leakage handling).

## Challenges Faced

**Severe class imbalance**: 
With only ~9% of customers churning, every model had to be evaluated on recall/ROC-AUC/F1 instead of accuracy, and class weights had to be threaded through manually for models like Gradient Boosting that don't accept class_weight directly.

**Inconsistent raw column naming**: 
The vbc_3g columns used month names instead of the numeric suffix convention used everywhere else, silently excluding them from month-grouping, leakage removal, and trend engineering until explicitly renamed — an easy-to-miss bug in a 226-column dataset.

**High dimensionality and missingness**: 
226 raw features with columns exceeding 40% missing values required a deliberate drop/impute strategy to balance signal retention against noise.

**Preventing target leakage**: 
Since churn was defined from month-9 behavior, every month-9 column had to be systematically identified and removed — a single missed column would have inflated metrics and made the model useless for real prospective scoring.

**Marginal returns from tuning and ensembling**: 
GB Tuned, LightGBM variants, and the Voting Ensemble all landed within ~0.002–0.006 AUC of the untuned Gradient Boosting baseline, meaning model selection ultimately hinged on recall/F1 tradeoffs rather than a clear AUC winner.

**Threshold selection was a genuine business tradeoff, not a technical one**: 
Choosing 0.45 required reasoning about the relative cost of a wasted email versus a missed churner — a decision that needed business input (which was gathered) rather than a purely statistical answer.

**Scoring-on-training-data caveat**: 
The final CHURN_FLAG export refits the model on the full dataset and scores that same data, which optimistically biases individual customer probabilities relative to true out-of-sample performance — an important limitation to disclose rather than a flaw to hide.

**Reattaching identifiers after feature engineering**: 
mobile_number was deliberately dropped before modeling (to avoid it being treated as a predictive feature), which meant it had to be carefully rejoined by index at export time — a step that's easy to get wrong if any rows are filtered during preprocessing.

## Future Work

**Cost-sensitive threshold optimization**: 
Instead of picking a single fixed threshold, incorporate actual retention-offer cost vs. customer lifetime value to choose the threshold that maximizes expected profit rather than F1.

**SHAP-based explainability**: 
Move beyond global feature importance (Gini/gain-based) to SHAP values for per-customer explanations, which would let a retention team understand why a specific high-value customer is flagged, not just that they are.

**Temporal validation**: 
Test whether a model trained on Jun–Aug generalizes to a different rolling 3-month window, to check robustness against seasonality rather than relying on a single random train/test split.

**Alternative churn definitions**: 
The current label (zero calls + zero recharge in month 9) is a strict "hard churn" definition; modeling partial disengagement (e.g., >50% usage drop without going to zero) could catch earlier-stage attrition and extend the intervention window.

**Deployment and monitoring**: 
Package the best model (or a distilled LightGBM version, given its speed advantage) behind a scoring API with drift monitoring, since telecom usage patterns and pricing plans shift over time and the model would need periodic retraining.

**Deeper handling of high-missingness features**: 
Rather than dropping columns >40% missing outright, investigate whether missingness itself is informative (e.g., missing recharge data could indicate a lapsed prepaid line) via missingness indicator flags.
Cost-aware ensembling: Since GB, LightGBM, and RF Tuned all sit close in AUC but differ meaningfully in precision/recall balance, a stacked ensemble (meta-learner on out-of-fold predictions) rather than simple soft voting might extract more value from their complementary error patterns.